# Modern Python Async: From Generators to =async/await=This final section connects the manual coroutine and scheduler machinery built earlier with Python's modern native `async`/`await` system (`asyncio`).  We proceed *side by side*: generator-based coroutines vs. native coroutines, manual scheduler vs. event loop, and `yield` vs. `await`.


### 1. Generator-Based Coroutine (Old Style) → Native Coroutine (Modern Python)


#### Old: coroutine as generator  A coroutine in the original lecture is simply a generator that *yields* to signal a suspension point.


In [ ]:
def child_old():    print("start child")    yield          # suspension    print("finish child")


#### New: coroutine as `async def`  A native coroutine suspends at an `await` point.


In [ ]:
async def child_new():    print("start child")    await asyncio.sleep(0)   # explicit suspension    print("finish child")


#### Concept Map  | Old Concept | New Concept ||-------------|-------------|| generator function | `async def` coroutine || `yield` to suspend | `await` to suspend || manual resume via `next()` | automatic via event loop |


### 2. Manual Scheduler → Asyncio Event Loop


#### Old: hand-written scheduler  The original material builds a scheduler by:  - keeping a run queue  - calling `next()` on coroutines  - removing completed ones


In [ ]:
# OLD: miniature cooperative schedulerfrom collections import dequedef run_old(tasks):    tasks = deque(tasks)    while tasks:        task = tasks.popleft()        try:            next(task)            tasks.append(task)        except StopIteration:            passdef childA_old():    print("A1"); yield    print("A2"); yielddef childB_old():    print("B1"); yield    print("B2"); yieldrun_old([childA_old(), childB_old()])


#### New: event loop  The `asyncio` event loop handles:  - scheduling  - task completion  - I/O readiness  - timers  - coroutine orchestration


In [ ]:
import asyncioasync def childA_new():    print("A1")    await asyncio.sleep(0)    print("A2")async def childB_new():    print("B1")    await asyncio.sleep(0)    print("B2")async def main_new():    await asyncio.gather(childA_new(), childB_new())asyncio.run(main_new())


#### Concept Map  | Old Scheduler | Modern `asyncio` ||---------------|------------------|| run queue | event loop internal queue || manual `next(coroutine)` | event loop invokes coroutine steps || generator terminates | `asyncio.Task` completes || custom system calls | builtin awaitables (`sleep`, `gather`, I/O) |


### 3. System Calls → Awaitables & Protocols

In the old model, a coroutine yields a *system call object* that the scheduler interprets:


In [ ]:
class GetTid: passdef task_old():    tid = yield GetTid()   # scheduler must decode this    print("my tid is", tid)    yield


#### New: `await` any object implementing the *Awaitable protocol*  Python formalizes this via:  - `__await__()` method  - or returning a native coroutine / Future


In [ ]:
class MyAwaitable:    def __await__(self):        print("awaiting MyAwaitable")        # nothing real here; minimal example        yield        return 42async def test_awaitable():    result = await MyAwaitable()    print("result =", result)import asyncioasyncio.run(test_awaitable())


#### Concept Map  | Old Feature | Modern Equivalent ||-------------|-------------------|| `yield SysCall` | `await Awaitable` || scheduler interprets yield | event loop calls `__await__` || designing ad-hoc syscall types | implement Awaitable / Future |


### 4. Putting It All Together

The modern system is *not new mechanics*—it is standardized notation over the same principles the lecture built from scratch.### Then vs. Now Summary| Idea | Old (Manual) | New (Native) ||------|--------------|--------------|| Coroutine | generator | `async def` || Suspend | `yield` | `await` || Resume | `next()` | event loop || Task | wrapper object | `asyncio.Task` || System call | yielded object | Awaitable/Future || Scheduler | user-written loop | `asyncio.run()` |


#### Final Words  Understanding the manual coroutine + scheduler model means you now understand *how Python async works internally*.  You can reason clearly about concurrency, avoid pitfalls, and design scalable async systems—like the sharded LLM server in the synthesis section.
